In [ ]:
pip install xgboost

In [2]:
import pandas as pd
import numpy as np
import xgboost as xgb

In [ ]:
hist_matches = pd.read_csv(r"data\results.csv", sep=',')
hist_matches['date'] = pd.to_datetime(hist_matches['date'])
hist_matches = hist_matches.iloc[:, 0:6].dropna() # droppa solo le ultime righe dei quarti del mondiale del 2026

print(hist_matches.shape)
hist_matches.head()

(49502, 6)


,date,home_team,away_team,home_score,away_score,tournament
0,1872-11-30,Scotland,England,0.0,0.0,Friendly
1,1873-03-08,England,Scotland,4.0,2.0,Friendly
2,1874-03-07,Scotland,England,2.0,1.0,Friendly
3,1875-03-06,England,Scotland,2.0,2.0,Friendly
4,1876-03-04,Scotland,England,3.0,0.0,Friendly


In [ ]:
teams_data_train = pd.read_csv(r"data\train.csv", sep=',') 
# non c'è il market value per il 2002
columns = list(range(0, 9)) + [10, 11, 13] # selecting only relevant columns
print(f"DataFrame total columns: {teams_data_train.shape[1]}")
teams_data_train = teams_data_train.iloc[:, columns]

print(teams_data_train.shape)
teams_data_train.head()

DataFrame total columns: 24
(192, 24)


,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,world_cup_titles_before,...,world_cup_participations_before,groups_passed_before,round16_before,quarterfinals_before,semifinals_before,finals_before,winner,finalist,semi_finalist,quarter_finalist
0,2006,Angola,Africa,0,61,49,19,13,14,0,...,0,0,0,0,0,0,0,0,0,0
1,2006,Argentina,South America,0,97,55,31,10,10,2,...,13,10,5,6,4,4,0,0,0,1
2,2006,Australia,Oceania,0,101,34,23,8,5,0,...,1,0,0,0,0,0,0,0,0,0
3,2006,Brazil,South America,0,117,47,30,9,17,5,...,17,15,7,11,9,6,0,0,0,1
4,2006,Costa Rica,North America,0,89,84,26,25,11,0,...,2,1,1,0,0,0,0,0,0,0


In [ ]:
teams_data_test = pd.read_csv(r"data\test.csv")
# sia nel train che test dataset presi non viene fatto dropna()
# perchè si perdono alcune righe per via di variabili poco importanti che non ci porteremo dietro
columns = list(range(0, 9)) + [10, 11, 13] # selecting only relevant columns
teams_data_test = teams_data_test.iloc[:, columns]

print(teams_data_test.shape)
teams_data_test.head()

(48, 12)


,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,squad_total_market_value_eur,fifa_rank_pre_tournament,squad_avg_age
0,2026,France,Europe,0,85,32,25,6,7,1290000000,1,26.7
1,2026,Spain,Europe,0,104,32,29,2,8,1150000000,2,27.2
2,2026,Argentina,South America,0,80,14,30,4,3,575000000,3,27.9
3,2026,England,Europe,0,82,23,26,6,7,1300000000,4,26.8
4,2026,Portugal,Europe,0,98,31,26,5,7,841000000,5,27.1


In [23]:
teams_data = pd.concat([teams_data_train, teams_data_test])
teams_data['version'] = pd.to_numeric(teams_data['version'])

teams_data.head()

,version,team,continent,is_host,goals_scored_last_4y,goals_received_last_4y,wins_last_4y,losses_last_4y,draws_last_4y,squad_total_market_value_eur,fifa_rank_pre_tournament,squad_avg_age
0,2006,Angola,Africa,0,61,49,19,13,14,52700000.0,57,29.2
1,2006,Argentina,South America,0,97,55,31,10,10,777200000.0,9,27.8
2,2006,Australia,Oceania,0,101,34,23,8,5,48030000.0,42,27.1
3,2006,Brazil,South America,0,117,47,30,9,17,858500000.0,1,27.1
4,2006,Costa Rica,North America,0,89,84,26,25,11,18400000.0,26,24.7


In [ ]:
world_cup_data = hist_matches[(hist_matches['date'] >= '2004-01-01') & (hist_matches['tournament'] == 'FIFA World Cup')]
world_cup_data['year'] = world_cup_data['date'].dt.year

world_cup_data = pd.merge(world_cup_data, teams_data, left_on=['year', 'home_team'], right_on=['version', 'team'], how='left').dropna().drop(columns=['version', 'team', 'continent']) # con dropna() levo le ultime tre partite e due righe che non so da dove le prende
world_cup_data = world_cup_data.rename(columns={'is_host':'home_host', 'goals_scored_last_4y':'home_scored_4',
       'goals_received_last_4y':'home_received_4', 'wins_last_4y':'home_wins_4', 'losses_last_4y':'home_loss_4',
       'draws_last_4y':'home_draws_4', 'squad_total_market_value_eur':'home_market_value(eur)',
       'fifa_rank_pre_tournament':'home_rank_pretournament', 'squad_avg_age':'home_avg_age'})

world_cup_data = pd.merge(world_cup_data, teams_data, left_on=['year', 'away_team'], right_on=['version', 'team'], how='left').dropna().drop(columns=[ 'version', 'team', 'continent'])
world_cup_data = world_cup_data.rename(columns={'is_host':'away_host', 'goals_scored_last_4y':'away_scored_4',
       'goals_received_last_4y':'away_received_4', 'wins_last_4y':'away_wins_4', 'losses_last_4y':'away_loss_4',
       'draws_last_4y':'away_draws_4', 'squad_total_market_value_eur':'away_market_value(eur)',
       'fifa_rank_pre_tournament':'away_rank_pretournament', 'squad_avg_age':'away_avg_age'})

world_cup_data['scored_4_delta'] = world_cup_data['home_scored_4'] - world_cup_data['away_scored_4']
world_cup_data['received_4_delta'] = world_cup_data['home_received_4'] - world_cup_data['away_received_4']
world_cup_data['wins_4_delta'] = world_cup_data['home_wins_4'] - world_cup_data['away_wins_4']
world_cup_data['loss_4_delta'] = world_cup_data['home_loss_4'] - world_cup_data['away_loss_4']
world_cup_data['draws_4_delta'] = world_cup_data['home_draws_4'] - world_cup_data['away_draws_4']
world_cup_data['market_value_delta(eur)'] = world_cup_data['home_market_value(eur)'] - world_cup_data['away_market_value(eur)']
world_cup_data['rank_pre_tournament_delta'] = world_cup_data['home_rank_pretournament'] - world_cup_data['away_rank_pretournament']
world_cup_data['avg_age_delta'] = world_cup_data['home_avg_age'] - world_cup_data['away_avg_age']

world_cup_data = world_cup_data.drop(columns=['home_scored_4', 'home_received_4', 'home_wins_4', 'home_loss_4',
       'home_draws_4', 'home_market_value(eur)', 'home_rank_pretournament', 'home_avg_age', 'away_scored_4',
       'away_received_4', 'away_wins_4', 'away_loss_4', 'away_draws_4', 'away_market_value(eur)',
       'away_rank_pretournament', 'away_avg_age'])

C:\Users\giang\AppData\Local\Temp\ipykernel_3084\2297569743.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  world_cup_data['year'] = world_cup_data['date'].dt.year


0      7.765000e+08
1      2.145000e+08
2      7.772000e+08
3      1.620000e+09
4      1.010000e+07
           ...     
410    8.410000e+08
411    1.697000e+08
412    5.750000e+08
413    2.510000e+08
414    1.290000e+09
Name: home_market_value(eur), Length: 411, dtype: float64

In [27]:
cond = [world_cup_data['home_score'] > world_cup_data['away_score'], world_cup_data['home_score'] == world_cup_data['away_score']]
val = [0, 1]

world_cup_data['target_col'] = np.select(cond, val, default=2)

features = world_cup_data[['year', 'home_host', 'away_host', 'scored_4_delta',
       'received_4_delta', 'wins_4_delta', 'loss_4_delta', 'draws_4_delta',
       'market_value_delta(eur)', 'rank_pre_tournament_delta', 'avg_age_delta']]
target = world_cup_data[['year','target_col']]
years = [2006, 2010, 2014, 2018, 2022]

world_cup_data.head()


,date,home_team,away_team,home_score,away_score,tournament,year,home_host,away_host,scored_4_delta,received_4_delta,wins_4_delta,loss_4_delta,draws_4_delta,market_value_delta(eur),rank_pre_tournament_delta,avg_age_delta,target_col
0,2006-06-09,Germany,Costa Rica,4.0,2.0,FIFA World Cup,2006,1.0,0.0,18.0,-18.0,0.0,-13.0,3.0,7.581000e+08,-7.0,3.0,0
1,2006-06-09,Poland,Ecuador,0.0,2.0,FIFA World Cup,2006,0.0,0.0,43.0,-19.0,15.0,-8.0,-5.0,-1.733300e+08,-10.0,1.9,2
2,2006-06-10,Argentina,Ivory Coast,2.0,1.0,FIFA World Cup,2006,0.0,0.0,30.0,21.0,10.0,2.0,2.0,3.663000e+08,-23.0,2.0,0
3,2006-06-10,England,Paraguay,1.0,0.0,FIFA World Cup,2006,0.0,0.0,30.0,-7.0,9.0,-5.0,-9.0,1.478150e+09,-23.0,-1.9,0
4,2006-06-10,Trinidad and Tobago,Sweden,0.0,0.0,FIFA World Cup,2006,0.0,0.0,-4.0,43.0,10.0,18.0,-7.0,-3.536800e+08,31.0,-1.7,1


In [ ]:
# Using XGBoost 
# Accuracy and loss are computed on the test set

from sklearn.metrics import accuracy_score, log_loss

fold_metrics = []

for y in years:

       X_train = features[(features['year'] != y)].drop(columns=['year'])
       Y_train = target[(target['year'] != y)]['target_col']

       X_test = features[(features['year'] == y)].drop(columns=['year']) # da prendere le colonne
       Y_test = target[(target['year'] == y)]['target_col']

       model = xgb.XGBClassifier(
              objective="multi:softprob", num_class=3,
              max_depth=3, min_child_weight=10,
              subsample=0.7, colsample_bytree=0.7,
              reg_lambda=2.0, n_estimators=300,
              eval_metric="mlogloss", random_state=42,
       )
       model.fit(X_train, Y_train)

       prob = model.predict_proba(X_test)
       preds = model.predict(X_test)

       acc = accuracy_score(Y_test, preds)
       loss = log_loss(Y_test, prob, labels=[0, 1, 2])
    
       fold_metrics.append({
              'Year': y,
              'Accuracy': acc,
              'LogLoss': loss
       })
    
       print(f"Mondiale {y} -> Accuracy: {acc:.4f} | LogLoss: {loss:.4f}")

Mondiale 2006 -> Accuracy: 0.4918 | LogLoss: 1.2780
Mondiale 2010 -> Accuracy: 0.4062 | LogLoss: 1.8517
Mondiale 2014 -> Accuracy: 0.4375 | LogLoss: 1.4120
Mondiale 2018 -> Accuracy: 0.4531 | LogLoss: 1.5816
Mondiale 2022 -> Accuracy: 0.4219 | LogLoss: 1.3493


In [67]:
X_train.shape, X_test.shape

((347, 10), (64, 10))

In [ ]:
# Computing accuracy and loss on X_train

fold_metrics = []

for y in years:

       X_train = features[(features['year'] != y)].drop(columns=['year'])
       Y_train = target[(target['year'] != y)]['target_col']

       X_test = features[(features['year'] == y)].drop(columns=['year']) # da prendere le colonne
       Y_test = target[(target['year'] == y)]['target_col']

       model = xgb.XGBClassifier(
              objective="multi:softprob", num_class=3,
              max_depth=3, min_child_weight=10,
              subsample=0.7, colsample_bytree=0.7,
              reg_lambda=2.0, n_estimators=300,
              eval_metric="mlogloss", random_state=42,
       )
       model.fit(X_train, Y_train)

       prob = model.predict_proba(X_train)
       preds = model.predict(X_train)

       acc = accuracy_score(Y_train, preds)
       loss = log_loss(Y_train, prob, labels=[0, 1, 2])
    
       fold_metrics.append({
              'Year': y,
              'Accuracy': acc,
              'LogLoss': loss
       })
    
       print(f"Mondiale {y} -> Accuracy: {acc:.4f} | LogLoss: {loss:.4f}")
       # it clearly overfits!!!
       # Definitevely beacuse of the scarsity of data and examples

Mondiale 2006 -> Accuracy: 0.9914 | LogLoss: 0.2237
Mondiale 2010 -> Accuracy: 0.9827 | LogLoss: 0.2029
Mondiale 2014 -> Accuracy: 0.9798 | LogLoss: 0.2222
Mondiale 2018 -> Accuracy: 0.9856 | LogLoss: 0.2117
Mondiale 2022 -> Accuracy: 0.9885 | LogLoss: 0.2107


In [ ]:
# Without for loop: training on 2002-2022 and testing on 2026
# Accuracy and loss are computed on the test set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['target_col']

X_test = features[(features['year'] == 2026)].drop(columns=['year']) # da prendere le colonne
Y_test = target[(target['year'] == 2026)]['target_col']

model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=2.0, n_estimators=300,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_test)
preds = model.predict(X_test)

acc = accuracy_score(Y_test, preds)
loss = log_loss(Y_test, prob, labels=[0, 1, 2])

print(f"Mondiale 2026 -> Accuracy: {acc:.4f} | LogLoss: {loss:.4f}")

Mondiale 2022 -> Accuracy: 0.4574 | LogLoss: 1.2708


In [ ]:
# Computing accuracy and loss on the training set

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['target_col']

X_test = features[(features['year'] == 2026)].drop(columns=['year']) # da prendere le colonne
Y_test = target[(target['year'] == 2026)]['target_col']

model = xgb.XGBClassifier(
        objective="multi:softprob", num_class=3,
        max_depth=3, min_child_weight=10,
        subsample=0.7, colsample_bytree=0.7,
        reg_lambda=2.0, n_estimators=300,
        eval_metric="mlogloss", random_state=42,
)
model.fit(X_train, Y_train)

prob = model.predict_proba(X_train)
preds = model.predict(X_train)

acc = accuracy_score(Y_train, preds)
loss = log_loss(Y_train, prob, labels=[0, 1, 2])

print(f"Mondiali 2006-2022 -> Accuracy: {acc:.4f} | LogLoss: {loss:.4f}")

# It clearly overfits this way as well, same problems 

Mondiale 2022 -> Accuracy: 0.9811 | LogLoss: 0.2403


In [ ]:
# Using a simpler model: Logistic Regression
# Accuracy and loss are computed on the training set

from sklearn.linear_model import LogisticRegression

fold_metrics = list()

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['target_col']

X_test = features[(features['year'] == 2026)].drop(columns=['year']) # da prendere le colonne
Y_test = target[(target['year'] == 2026)]['target_col']

model = LogisticRegression(max_iter=200)
_ = model.fit(X_train, Y_train)

prob = _.predict_proba(X_train)
preds = _.predict(X_train)

acc = accuracy_score(Y_train, preds)
loss = log_loss(Y_train, prob, labels=[0, 1, 2])

fold_metrics.append({
    "accuracy": acc, "log_loss": loss
})

print(f"Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

# Accuracy significantly drops, while loss is lower
# Why???

Accuracy: 0.5584, LogLoss: 0.9838


In [ ]:
# computing accuracy and loss on test set

from sklearn.linear_model import LogisticRegression

fold_metrics = list()

X_train = features[(features['year'] != 2026)].drop(columns=['year'])
Y_train = target[(target['year'] != 2026)]['target_col']

X_test = features[(features['year'] == 2026)].drop(columns=['year']) # da prendere le colonne
Y_test = target[(target['year'] == 2026)]['target_col']

model = LogisticRegression(max_iter=200)
_ = model.fit(X_train, Y_train)

prob = _.predict_proba(X_test)
preds = _.predict(X_test)

acc = accuracy_score(Y_test, preds)
loss = log_loss(Y_test, prob, labels=[0, 1, 2])

fold_metrics.append({
    "accuracy": acc, "log_loss": loss
})

print(f"Accuracy: {acc:.4f}, LogLoss: {loss:.4f}")

Accuracy: 0.6809, LogLoss: 0.9011
